In [2]:
import pandas as pd
import time
import math

df = pd.read_csv("sep.csv").sample(n=50_000, random_state=42)
df.to_csv("sep_sample.csv", index=False)

df = pd.read_csv("sep_sample.csv")
# Rūšiuojame reitingo stulpelį (efektyviems paieškos algoritmams reikalinga)
data = sorted(df['WhiteElo'].tolist())
print('Kiek yra duomenyse:', len(data))



Kiek yra duomenyse: 50000


Task 1: Search algorithms

In [3]:
def linear_search(arr, x):
    for i in range(len(arr)):
        if arr[i] == x: return i
    return -1

def binary_search(arr, x):
    low, high = 0, len(arr) - 1
    while low <= high:
        mid = (low + high) // 2
        if arr[mid] == x: return mid
        elif arr[mid] < x: low = mid + 1
        else: high = mid - 1
    return -1

def jump_search(arr, x):
    n = len(arr)
    step = int(math.sqrt(n))
    prev = 0
    while prev < n and arr[min(step, n) - 1] < x:
        prev = step
        step += int(math.sqrt(n))
    for i in range(prev, min(step, n)):
        if arr[i] == x: return i
    return -1

def interpolation_search(arr, x):
    low, high = 0, len(arr) - 1
    while low <= high and x >= arr[low] and x <= arr[high]:
        if low == high: return low if arr[low] == x else -1
        pos = low + int(((high - low) / (arr[high] - arr[low])) * (x - arr[low]))
        if arr[pos] == x: return pos
        if arr[pos] < x: low = pos + 1
        else: high = pos - 1
    return -1

sizes = [1000, 10000, 50000]
algos = [
    ("Linear Search", linear_search),
    ("Binary Search", binary_search),
    ("Jump Search", jump_search),
    ("Interpolation Search", interpolation_search)
]

sizes = [1000, 10000, 50000]
algos = [
    ("Linear Search", linear_search),
    ("Binary Search", binary_search),
    ("Jump Search", jump_search),
    ("Interpolation Search", interpolation_search)
]

for name, func in algos:
    print(f"Rezultatai: {name}")
    for n in sizes:
        subset = data[:n]
        target = subset[-1] # Ieškome paskutinio elemento (blogiausias atvejis)
        
        # Matome laiką 3 kartus ir imame vidurkį (kaip sąlygoje)
        t_sum = 0
        for _ in range(3):
            start = time.perf_counter()
            func(subset, target)
            t_sum += (time.perf_counter() - start)
        
        avg_ms = (t_sum / 3) * 1000 # Konvertuojame į milisekundes
        print(f"n = {n}: {avg_ms:.6f} ms")


Rezultatai: Linear Search
n = 1000: 0.256233 ms
n = 10000: 2.242133 ms
n = 50000: 11.256500 ms
Rezultatai: Binary Search
n = 1000: 0.008033 ms
n = 10000: 0.005333 ms
n = 50000: 0.007033 ms
Rezultatai: Jump Search
n = 1000: 0.047600 ms
n = 10000: 0.076400 ms
n = 50000: 0.159633 ms
Rezultatai: Interpolation Search
n = 1000: 0.011067 ms
n = 10000: 0.003033 ms
n = 50000: 0.008067 ms


Kai n = 1 000, geriausius rezultatus rodo Binary Search ir Interpolation Search.

Kai n = 50 000, laimi Interpolation Search. Jis greičiausias, nes duomenys yra gana tolygiai paskirstyti.

In [4]:
# Blogiausias atvejis Interpoliacinei paieškai
# Skaičiai auga labai netolygiai (dvejeto laipsniu)
bad_input = [2**i for i in range(100)]
target = bad_input[-2]

print(f"Ieškoma reikšmė: {target}")

# Binarinė paieška
t1 = time.perf_counter()
binary_search(bad_input, target)
t2 = time.perf_counter()
print(f"Binarinė paieška: {(t2 - t1) * 1000:.6f} ms")

# Interpoliacinė paieška
t1 = time.perf_counter()
interpolation_search(bad_input, target)
t2 = time.perf_counter()
print(f"Interpoliacinė paieška: {(t2 - t1) * 1000:.6f} ms")

Ieškoma reikšmė: 316912650057057350374175801344
Binarinė paieška: 0.206700 ms
Interpoliacinė paieška: 0.334000 ms


Nors Interpoliacinė paieška yra itin greita su tolygiais duomenimis, ji yra nepatikima, kai duomenų pasiskirstymas yra netolygus ar turi didelių nuokrypių. Tokiu atveju Binarinė paieška yra pranašesnė.

Task 2: Binary Search Tree (BST) and traversals

In [5]:
from collections import deque

#BST Mazgas ir Klasė
class Node:
    def __init__(self, key, value):
        self.key = key
        self.value = value
        self.left = None
        self.right = None

class BinarySearchTree:
    def __init__(self):
        self.root = None

    def insert(self, key, value):
        if not self.root:
            self.root = Node(key, value)
            return
        curr = self.root
        while True:
            if key < curr.key:
                if curr.left: curr = curr.left
                else: 
                    curr.left = Node(key, value)
                    break
            else:
                if curr.right: curr = curr.right
                else: 
                    curr.right = Node(key, value)
                    break

    def search(self, key):
        curr = self.root
        while curr:
            if key == curr.key: return curr.value
            curr = curr.left if key < curr.key else curr.right
        return None

    def delete(self, key):
        # Naudojame paprastą rekursiją trynimui (visas 3 sąlygas apdorojantis būdas)
        def _delete(node, k):
            if not node: return None
            if k < node.key: node.left = _delete(node.left, k)
            elif k > node.key: node.right = _delete(node.right, k)
            else:
                if not node.left: return node.right
                if not node.right: return node.left
                # Mazgas su dviem vaikais, ieškome mažiausio dešinėje
                temp = node.right
                while temp.left: temp = temp.left
                node.key, node.value = temp.key, temp.value
                node.right = _delete(node.right, temp.key)
            return node
        self.root = _delete(self.root, key)

    # Apėjimai (Recursion)
    def inorder(self, node):
        if node:
            self.inorder(node.left)
            print(node.key, end=" ")
            self.inorder(node.right)

    # BFS (Queue)
    def bfs(self):
        if not self.root: return
        q = deque([self.root])
        while q:
            node = q.popleft()
            # print(node.key) # Galima atspausdinti testavimui
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)

    # 3. Range Query
    def range_query(self, low, high):
        res = []
        def _range(node):
            if not node: return
            if low < node.key: _range(node.left)
            if low <= node.key <= high: res.append(node.value)
            if high > node.key: _range(node.right)
        _range(self.root)
        return res

# Duomenų užkrovimas ir Medžio kūrimas
df = pd.read_csv("sep_sample.csv").head(50000)
bst = BinarySearchTree()

# Įkeliame duomenis (key = WhiteElo, value = visa eilutė)
data_list = df.to_dict('records')
for row in data_list:
    bst.insert(row['WhiteElo'], row)

#Range Query palyginimas (Benchmark)
low, high = 2500, 2700 # Parinkime dažnesnį intervalą, kad rastų rezultatų

# BST laikas
t1 = time.perf_counter()
res_bst = bst.range_query(low, high)
t2 = time.perf_counter()
print(f"BST laikas: {(t2-t1)*1000:.4f} ms")

# Lazy Baseline laikas
t1 = time.perf_counter()
res_lazy = [r for r in data_list if low <= r['WhiteElo'] <= high]
t2 = time.perf_counter()
print(f"Lazy laikas: {(t2-t1)*1000:.4f} ms")
print(f"Rasta: {len(res_bst)}")

BST laikas: 18.3488 ms
Lazy laikas: 20.4759 ms
Rasta: 12018


Task 3: Heap and top-k

In [6]:

# Heap įgyvendinimas naudojant masyvą (be heapq importo)
class MinHeap:
    def __init__(self, capacity):
        self.heap = []
        self.capacity = capacity

    def _parent(self, i): return (i - 1) // 2
    def _left(self, i): return 2 * i + 1
    def _right(self, i): return 2 * i + 2

    def insert(self, value):
        # Įterpiame į galą ir "keliame" į viršų (bubble up)
        self.heap.append(value)
        self._bubble_up(len(self.heap) - 1)
        # Jei viršijame k=10, pašaliname mažiausią (šaknį)
        if len(self.heap) > self.capacity:
            self.extract()

    def extract(self):
        if len(self.heap) == 0: return None
        if len(self.heap) == 1: return self.heap.pop()
        
        root = self.heap[0]
        self.heap[0] = self.heap.pop()
        self._bubble_down(0)
        return root

    def peek(self):
        return self.heap[0] if self.heap else None

    def _bubble_up(self, i):
        while i > 0 and self.heap[i] < self.heap[self._parent(i)]:
            self.heap[i], self.heap[self._parent(i)] = self.heap[self._parent(i)], self.heap[i]
            i = self._parent(i)

    def _bubble_down(self, i):
        smallest = i
        l, r = self._left(i), self._right(i)
        if l < len(self.heap) and self.heap[l] < self.heap[smallest]: smallest = l
        if r < len(self.heap) and self.heap[r] < self.heap[smallest]: smallest = r
        if smallest != i:
            self.heap[i], self.heap[smallest] = self.heap[smallest], self.heap[i]
            self._bubble_down(smallest)

# Top-k vykdymas ir palyginimas
df = pd.read_csv("sep_sample.csv").head(50000)
ratings = df['WhiteElo'].tolist()
k = 10

# HEAP METODAS
t1 = time.perf_counter()
min_heap = MinHeap(k)
for r in ratings:
    # Jei krūva nepilna arba naujas reitingas didesnis už mažiausią krūvoje
    if len(min_heap.heap) < k or r > min_heap.peek():
        min_heap.insert(r)
heap_res = sorted(min_heap.heap, reverse=True)
t2 = time.perf_counter()
heap_time = (t2 - t1) * 1000

# NAIVE BASELINE (Rūšiavimas)
t1 = time.perf_counter()
naive_res = sorted(ratings, reverse=True)[:k]
t2 = time.perf_counter()
naive_time = (t2 - t1) * 1000

# Rezultatų spausdinimas
print(f"Top-10 reitingai: {heap_res}")
print(f"Heap laikas (n=50,000): {heap_time:.4f} ms")
print(f"Naive laikas (n=50,000): {naive_time:.4f} ms")

Top-10 reitingai: [2976, 2968, 2966, 2961, 2878, 2860, 2858, 2854, 2854, 2851]
Heap laikas (n=50,000): 19.1331 ms
Naive laikas (n=50,000): 10.9302 ms


Kodėl Heap tinka top-k?
Krūva (Heap) visada mato tik patį viršų (didžiausią arba mažiausią elementą). Mums nereikia rūšiuoti visų 50 000 skaičių

Kodėl „Heap“ netinka rasti elementą x?
Krūvoje nėra tvarkos tarp kairės ir dešinės pusės (tik tarp tėvų ir vaikų). Jei ieškome konkretaus skaičiaus viduryje, krūva mums nepadeda